# Backtest Visualization Dashboard

**REBUILT:** 2026-04-06  
**Purpose:** default levered vs unlevered comparison, net-of-cost equity curves, financing impact, and exposure sensitivity.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.figsize': (16, 7),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
})

BASE = Path('..')
REPORTS = BASE / 'reports'
SWEEP_DIR = BASE / 'report_snapshots' / 'exposure_sensitivity_2026-04-06'
UNLEV_DIR = BASE / 'report_snapshots' / 'unlevered_profile_2026-04-06'

bt = pd.read_csv(REPORTS / 'backtest_results.csv')
bt['Date'] = pd.to_datetime(bt['Date'])
bt = bt.sort_values('Date').reset_index(drop=True)
bt['StrategyRetNet'] = bt['PortRetKS'] if 'PortRetKS' in bt.columns else bt['PortRet']
bt['StrategyEqNet'] = (1 + bt['StrategyRetNet']).cumprod()
bt['StrategyEqGross'] = (1 + bt['GrossRet']).cumprod()
bt['SPYEq'] = (1 + bt['SPYRet']).cumprod()

unlev = pd.read_csv(UNLEV_DIR / 'backtest_results.csv')
unlev['Date'] = pd.to_datetime(unlev['Date'])
unlev = unlev.sort_values('Date').reset_index(drop=True)
unlev['StrategyRetNet'] = unlev['PortRetKS'] if 'PortRetKS' in unlev.columns else unlev['PortRet']
unlev['StrategyEqNet'] = (1 + unlev['StrategyRetNet']).cumprod()
unlev['StrategyEqGross'] = (1 + unlev['GrossRet']).cumprod()

turnover_text = (REPORTS / 'turnover_report.txt').read_text(encoding='utf-8') if (REPORTS / 'turnover_report.txt').exists() else ''
reliability_text = (REPORTS / 'reliability_report.txt').read_text(encoding='utf-8') if (REPORTS / 'reliability_report.txt').exists() else ''
quarter_log_text = (REPORTS / 'quarter_log.txt').read_text(encoding='utf-8') if (REPORTS / 'quarter_log.txt').exists() else ''

pct_fmt = FuncFormatter(lambda y, _: f'{y:.0%}')

COLORS = {
    'levered': '#0f766e',
    'unlevered': '#b45309',
    'spy': '#334155',
    'gross': '#2563eb',
    'cost': '#7c3aed',
    '1.10': '#94a3b8',
    '1.20': '#0f766e',
    '1.30': '#2563eb',
    '1.50': '#7c3aed',
}

def pct(x, d=2):
    return '' if pd.isna(x) else f'{x:.{d}%}'

def num(x, d=2):
    return '' if pd.isna(x) else f'{x:.{d}f}'

def max_dd(eq):
    eq = pd.Series(eq).dropna()
    return np.nan if eq.empty else float((eq / eq.cummax() - 1).min())

def cagr(returns, periods=4):
    r = pd.Series(returns).dropna()
    if r.empty:
        return np.nan
    total = float((1 + r).prod())
    years = len(r) / periods
    return total ** (1 / years) - 1 if years else np.nan

def ann_vol(returns, periods=4):
    r = pd.Series(returns).dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std(ddof=1) * np.sqrt(periods))

def sharpe(returns, periods=4):
    r = cagr(returns, periods)
    v = ann_vol(returns, periods)
    return np.nan if pd.isna(v) or v == 0 else r / v

def sortino(returns, periods=4):
    r = pd.Series(returns).dropna()
    downside = r[r < 0]
    if len(downside) < 2:
        return np.nan
    down_vol = float(downside.std(ddof=1) * np.sqrt(periods))
    cg = cagr(r, periods)
    return np.nan if down_vol == 0 else cg / down_vol

def calmar(returns, equity, periods=4):
    cg = cagr(returns, periods)
    dd = abs(max_dd(equity))
    return np.nan if dd == 0 or pd.isna(dd) else cg / dd

def metrics_row(label, returns, equity, exec_bps=np.nan, fin_bps=np.nan):
    r = pd.Series(returns).dropna()
    return {
        'Profile': label,
        'Total Return': float((1 + r).prod() - 1) if not r.empty else np.nan,
        'CAGR': cagr(r),
        'Ann Vol': ann_vol(r),
        'Sharpe': sharpe(r),
        'Sortino': sortino(r),
        'Max Drawdown': max_dd(equity),
        'Calmar': calmar(r, equity),
        'Hit Rate': float((r > 0).mean()) if not r.empty else np.nan,
        'Best Quarter': float(r.max()) if not r.empty else np.nan,
        'Worst Quarter': float(r.min()) if not r.empty else np.nan,
        'Avg Exec Cost (bps/q)': exec_bps,
        'Avg Financing Cost (bps/q)': fin_bps,
    }

def summarize_profile(label, df):
    net = df['PortRetKS'] if 'PortRetKS' in df.columns else df['PortRet']
    eq = (1 + net).cumprod()
    return {
        'Profile': label,
        'TotalReturn': float(eq.iloc[-1] - 1) if len(eq) else np.nan,
        'AnnReturn': cagr(net),
        'AnnVol': ann_vol(net),
        'Sharpe': sharpe(net),
        'MaxDrawdown': max_dd(eq),
        'AvgExecutionCostBpsQ': float(df['TxnCostBps'].mean()) if 'TxnCostBps' in df.columns else np.nan,
        'AvgFinancingCostBpsQ': float(df['FinancingCostRet'].mean() * 1e4) if 'FinancingCostRet' in df.columns else np.nan,
    }

def build_exposure_sweep_df():
    rows = []
    for path in sorted(SWEEP_DIR.glob('tmp_*/backtest_results.csv')):
        exposure_txt = path.parent.name.replace('tmp_', '').replace('p', '.')
        exposure = float(exposure_txt)
        df = pd.read_csv(path).sort_values('Date').reset_index(drop=True)
        net = df['PortRetKS'] if 'PortRetKS' in df.columns else df['PortRet']
        eq = (1 + net).cumprod()
        rows.append({
            'ExpansionExposure': exposure,
            'TotalReturn': float(eq.iloc[-1] - 1) if len(eq) else np.nan,
            'FinalEquity': float(eq.iloc[-1]) if len(eq) else np.nan,
            'AnnReturn': cagr(net),
            'AnnVol': ann_vol(net),
            'Sharpe': sharpe(net),
            'MaxDrawdown': max_dd(eq),
            'HitRate': float((net > 0).mean()) if len(net) else np.nan,
            'AvgQuarterRet': float(net.mean()) if len(net) else np.nan,
            'AvgExpansionQuarter': float(net[df['Regime'] == 'Expansion'].mean()) if 'Regime' in df.columns else np.nan,
            'AvgTransitionQuarter': float(net[df['Regime'] == 'Transition'].mean()) if 'Regime' in df.columns else np.nan,
            'AvgContractionQuarter': float(net[df['Regime'] == 'Contraction'].mean()) if 'Regime' in df.columns else np.nan,
            'AvgFinancingCostBpsQ': float(df['FinancingCostRet'].mean() * 1e4) if 'FinancingCostRet' in df.columns else np.nan,
            'AvgFinancingRateAnnual': float(df.loc[df['FinancingRateAnnual'] > 0, 'FinancingRateAnnual'].mean()) if 'FinancingRateAnnual' in df.columns else np.nan,
            'SPYTotalReturn': float((1 + df['SPYRet']).prod() - 1) if 'SPYRet' in df.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values('ExpansionExposure').reset_index(drop=True)

compare_df = pd.DataFrame([
    summarize_profile('Levered 1.20 / 1.00 / 0.50', bt),
    summarize_profile('Unlevered 1.00 / 0.80 / 0.50', unlev),
])
sweep_df = build_exposure_sweep_df()


## Summary Cards


In [ ]:
lev = compare_df.loc[compare_df['Profile'].str.contains('Levered')].iloc[0]
unl = compare_df.loc[compare_df['Profile'].str.contains('Unlevered')].iloc[0]

def card(title, value, note):
    return f"<div style='padding:16px 18px;border:1px solid #d7dee7;border-radius:14px;background:#f8fafc;min-width:210px'><div style='font-size:12px;text-transform:uppercase;letter-spacing:.08em;color:#64748b'>{title}</div><div style='font-size:30px;font-weight:700;color:#0f172a;margin-top:8px'>{value}</div><div style='font-size:12px;color:#64748b;margin-top:8px'>{note}</div></div>"

cards = [
    card('Levered Total Return', pct(lev['TotalReturn']), 'Current default profile'),
    card('Levered CAGR', pct(lev['AnnReturn']), 'Net of execution and financing'),
    card('Unlevered Total Return', pct(unl['TotalReturn']), 'Expansion 1.00 / Transition 0.80'),
    card('Unlevered CAGR', pct(unl['AnnReturn']), 'No leverage profile'),
    card('Financing Drag', f"{lev['AvgFinancingCostBpsQ']:.2f} bps/q", 'Average levered financing cost'),
    card('CAGR Gap', pct(lev['AnnReturn'] - unl['AnnReturn']), 'Levered minus unlevered'),
]
display(HTML("<div style='display:flex;flex-wrap:wrap;gap:14px'>" + ''.join(cards) + "</div>"))


## Levered vs Unlevered Performance Metrics

This is the explicit net-of-cost metrics table for the two profiles, plus SPY.

In [ ]:
perf = pd.DataFrame([
    metrics_row('Levered 1.20 / 1.00 / 0.50', bt['StrategyRetNet'], bt['StrategyEqNet'], float(bt['TxnCostBps'].mean()), float(bt['FinancingCostRet'].mean() * 1e4)),
    metrics_row('Unlevered 1.00 / 0.80 / 0.50', unlev['StrategyRetNet'], unlev['StrategyEqNet'], float(unlev['TxnCostBps'].mean()), float(unlev['FinancingCostRet'].mean() * 1e4)),
    metrics_row('SPY', bt['SPYRet'], bt['SPYEq']),
])

perf_show = perf.copy()
for c in ['Total Return', 'CAGR', 'Ann Vol', 'Max Drawdown', 'Hit Rate', 'Best Quarter', 'Worst Quarter']:
    perf_show[c] = perf_show[c].map(pct)
for c in ['Sharpe', 'Sortino', 'Calmar', 'Avg Exec Cost (bps/q)', 'Avg Financing Cost (bps/q)']:
    perf_show[c] = perf_show[c].map(num)
display(perf_show)


## Net Equity Curves

These curves are net of costs. Levered includes execution plus financing. Unlevered includes execution only.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(17, 11), sharex=True)

axes[0].plot(bt['Date'], bt['StrategyEqNet'], color=COLORS['levered'], linewidth=2.7, label='Levered Net')
axes[0].plot(unlev['Date'], unlev['StrategyEqNet'], color=COLORS['unlevered'], linewidth=2.7, label='Unlevered Net')
axes[0].plot(bt['Date'], bt['SPYEq'], color=COLORS['spy'], linewidth=2.1, label='SPY')
axes[0].set_title('Net Equity Curves')
axes[0].set_ylabel('Growth of $1')
axes[0].legend(loc='upper left')

lev_dd = bt['StrategyEqNet'] / bt['StrategyEqNet'].cummax() - 1
unl_dd = unlev['StrategyEqNet'] / unlev['StrategyEqNet'].cummax() - 1
axes[1].plot(bt['Date'], lev_dd, color=COLORS['levered'], linewidth=2.4, label='Levered DD')
axes[1].plot(unlev['Date'], unl_dd, color=COLORS['unlevered'], linewidth=2.4, label='Unlevered DD')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].yaxis.set_major_formatter(pct_fmt)
axes[1].set_title('Drawdown Comparison')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()


## Cost Impact On Levered Strategy

This makes the friction impact explicit: gross strategy, net strategy, and the cumulative gap caused by execution plus financing.

In [ ]:
cost_gap = bt['StrategyEqGross'] - bt['StrategyEqNet']

fig, axes = plt.subplots(1, 2, figsize=(19, 6))
axes[0].plot(bt['Date'], bt['StrategyEqGross'], color=COLORS['gross'], linewidth=2.5, label='Gross Strategy')
axes[0].plot(bt['Date'], bt['StrategyEqNet'], color=COLORS['levered'], linewidth=2.7, label='Net Levered Strategy')
axes[0].set_title('Gross vs Net Levered Equity')
axes[0].set_ylabel('Growth of $1')
axes[0].legend(loc='upper left')

axes[1].fill_between(bt['Date'], cost_gap, color=COLORS['cost'], alpha=0.25)
axes[1].plot(bt['Date'], cost_gap, color=COLORS['cost'], linewidth=2.4)
axes[1].set_title('Cumulative Cost Gap: Gross Minus Net')
axes[1].set_ylabel('Equity Gap')

plt.tight_layout()
plt.show()


## Costs And Financing Summary


In [ ]:
cost_summary = pd.DataFrame({
    'Metric': [
        'Average Turnover',
        'Average Execution Cost (bps/q)',
        'Average Financing Cost (bps/q)',
        'Average Borrowed Notional (USD)',
        'Average Financing Rate',
        'Average Holdings',
    ],
    'Levered': [
        pct(bt['Turnover'].mean()),
        num(bt['TxnCostBps'].mean()),
        num(bt['FinancingCostRet'].mean() * 1e4),
        f"${bt['BorrowedNotionalUSD'].mean():,.0f}",
        pct(bt.loc[bt['FinancingRateAnnual'] > 0, 'FinancingRateAnnual'].mean()),
        num(bt['NHoldings'].mean()),
    ],
    'Unlevered': [
        pct(unlev['Turnover'].mean()),
        num(unlev['TxnCostBps'].mean()),
        num(unlev['FinancingCostRet'].mean() * 1e4),
        f"${unlev['BorrowedNotionalUSD'].mean():,.0f}",
        pct(unlev.loc[unlev['FinancingRateAnnual'] > 0, 'FinancingRateAnnual'].mean()),
        num(unlev['NHoldings'].mean()),
    ]
})
display(cost_summary)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(bt['Date'], bt['Turnover'], color='#2563eb', linewidth=2.2)
axes[0].set_title('Turnover')
axes[0].yaxis.set_major_formatter(pct_fmt)

axes[1].plot(bt['Date'], bt['TxnCostBps'], color='#b45309', linewidth=2.2)
axes[1].set_title('Execution Cost (bps)')

axes[2].plot(bt['Date'], bt['FinancingCostRet'] * 1e4, color='#7c3aed', linewidth=2.2)
axes[2].set_title('Financing Drag (bps)')

plt.tight_layout()
plt.show()


## Expansion Exposure Sensitivity

These are the financed sweep results for different Expansion exposures, with the unlevered profile as a reference line.

In [ ]:
sweep_show = sweep_df.copy()
for c in ['TotalReturn', 'AnnReturn', 'AnnVol', 'MaxDrawdown', 'HitRate', 'AvgQuarterRet', 'AvgExpansionQuarter', 'AvgTransitionQuarter', 'AvgContractionQuarter', 'AvgFinancingRateAnnual']:
    if c in sweep_show.columns:
        sweep_show[c] = sweep_show[c].map(pct)
for c in ['ExpansionExposure', 'FinalEquity', 'Sharpe', 'AvgFinancingCostBpsQ']:
    if c in sweep_show.columns:
        sweep_show[c] = sweep_show[c].map(num)
display(sweep_show)


In [ ]:
sweep_files = {
    1.10: SWEEP_DIR / 'backtest_results_expansion_1p10.csv',
    1.20: SWEEP_DIR / 'backtest_results_expansion_1p20.csv',
    1.30: SWEEP_DIR / 'backtest_results_expansion_1p30.csv',
    1.50: SWEEP_DIR / 'backtest_results_expansion_1p50.csv',
}

fig, ax = plt.subplots(figsize=(17, 7))
for exposure in [1.10, 1.20, 1.30, 1.50]:
    df = pd.read_csv(sweep_files[exposure])
    df['Date'] = pd.to_datetime(df['Date'])
    r = df['PortRetKS'] if 'PortRetKS' in df.columns else df['PortRet']
    eq = (1 + r).cumprod()
    ax.plot(df['Date'], eq, linewidth=2.3, color=COLORS[f'{exposure:.2f}'], label=f'Expansion {exposure:.2f}x')

ax.plot(unlev['Date'], unlev['StrategyEqNet'], linewidth=2.6, linestyle='--', color=COLORS['unlevered'], label='Unlevered 1.00 / 0.80 / 0.50')
ax.set_title('Equity Curves Across Expansion Exposure Settings')
ax.set_ylabel('Growth of $1')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()


## Regime Breakdown


In [ ]:
regime_perf = bt.groupby('Regime', dropna=False).agg(
    Quarters=('Date', 'count'),
    AvgReturn=('StrategyRetNet', 'mean'),
    AvgExposure=('RegimeExp', 'mean'),
    AvgTurnover=('Turnover', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.barplot(data=regime_perf, x='Regime', y='Quarters', ax=axes[0], palette='crest')
axes[0].set_title('Regime Distribution')

sns.barplot(data=regime_perf, x='Regime', y='AvgReturn', ax=axes[1], palette='flare')
axes[1].set_title('Average Net Return By Regime')
axes[1].yaxis.set_major_formatter(pct_fmt)

sns.barplot(data=regime_perf, x='Regime', y='AvgExposure', ax=axes[2], palette='mako')
axes[2].set_title('Average Exposure By Regime')

plt.tight_layout()
plt.show()

regime_show = regime_perf.copy()
for c in ['AvgReturn', 'AvgTurnover']:
    regime_show[c] = regime_show[c].map(pct)
regime_show['AvgExposure'] = regime_show['AvgExposure'].map(num)
display(regime_show)


## Quarter Path And Extremes


In [ ]:
rolling = bt[['Date', 'StrategyRetNet', 'SPYRet']].copy()
rolling['StrategyRolling1Y'] = rolling['StrategyRetNet'].rolling(4).apply(lambda x: np.prod(1 + x) - 1, raw=True)
rolling['SPYRolling1Y'] = rolling['SPYRet'].rolling(4).apply(lambda x: np.prod(1 + x) - 1, raw=True)

fig, axes = plt.subplots(1, 2, figsize=(19, 6))
axes[0].plot(rolling['Date'], rolling['StrategyRolling1Y'], color=COLORS['levered'], linewidth=2.2, label='Strategy')
axes[0].plot(rolling['Date'], rolling['SPYRolling1Y'], color=COLORS['spy'], linewidth=2.0, label='SPY')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].yaxis.set_major_formatter(pct_fmt)
axes[0].set_title('Rolling 1-Year Return')
axes[0].legend(loc='upper left')

bar_colors = np.where(bt['StrategyRetNet'] >= 0, COLORS['levered'], '#dc2626')
axes[1].bar(bt['Date'], bt['StrategyRetNet'], width=65, color=bar_colors, alpha=0.85)
axes[1].plot(bt['Date'], bt['SPYRet'], color=COLORS['spy'], linewidth=2.0)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].yaxis.set_major_formatter(pct_fmt)
axes[1].set_title('Quarterly Net Returns')

plt.tight_layout()
plt.show()

ext = bt[['Date', 'StrategyRetNet', 'SPYRet', 'Regime', 'PredSectorNames', 'RealTopSectors']].copy()
ext = ext.sort_values('StrategyRetNet', ascending=False)

def fmt(df):
    out = df.copy()
    out['Date'] = pd.to_datetime(out['Date']).dt.date.astype(str)
    out['StrategyRetNet'] = out['StrategyRetNet'].map(pct)
    out['SPYRet'] = out['SPYRet'].map(pct)
    return out

display(Markdown('### Best 10 Quarters'))
display(fmt(ext.head(10)))
display(Markdown('### Worst 10 Quarters'))
display(fmt(ext.tail(10).sort_values('StrategyRetNet')))


## Report Excerpts

This section is safe to run. It prints the latest report text without the broken newline issue from the previous notebook.

In [ ]:
display(Markdown('### Reliability Report'))
print(reliability_text if reliability_text.strip() else 'No reliability report found.')

display(Markdown('### Turnover / Financing Report'))
print(turnover_text if turnover_text.strip() else 'No turnover report found.')

display(Markdown('### Quarter Log Preview'))
print('\n'.join(quarter_log_text.splitlines()[:40]) if quarter_log_text.strip() else 'No quarter log found.')
